In [7]:
# Imports
import tensorflow as tf
import os
import sys


from preprocess import create_data_generators, get_class_map

# import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

from pathlib import Path




In [8]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20

In [9]:
from pathlib import Path
import os

PROJECT_BASE_DIR = Path(os.environ["NEURODETECT_BASE_DIR"])

TRAIN_DIR = str(PROJECT_BASE_DIR / "train")
TEST_DIR = str(PROJECT_BASE_DIR / "test")
MODELS_DIR = PROJECT_BASE_DIR / "models"

print("Project base:", PROJECT_BASE_DIR)
print("Train path:", TRAIN_DIR)
print("Test path:", TEST_DIR)
print("Models path:", MODELS_DIR)

print("Train exists:", os.path.exists(TRAIN_DIR))
print("Test exists:", os.path.exists(TEST_DIR))
print("Models exists:", os.path.exists(MODELS_DIR))

Project base: G:\My Drive\SeniorProject\datasets\dataset
Train path: G:\My Drive\SeniorProject\datasets\dataset\train
Test path: G:\My Drive\SeniorProject\datasets\dataset\test
Models path: G:\My Drive\SeniorProject\datasets\dataset\models
Train exists: True
Test exists: True
Models exists: True


In [10]:
train_generator, test_generator = create_data_generators(
    TRAIN_DIR,
    TEST_DIR,
    IMG_SIZE,
    BATCH_SIZE
)

labels = get_class_map(train_generator)

Found 2870 images belonging to 4 classes.
Found 394 images belonging to 4 classes.


In [11]:

X_batch, y_batch = next(train_generator) 
print("Batch image shape:", X_batch.shape)  

print("\nLoading Test Data:")

X_batch, y_batch = next(test_generator)  
print("Batch image shape:", X_batch.shape) 


Batch image shape: (32, 224, 224, 3)

Loading Test Data:
Batch image shape: (32, 224, 224, 3)


In [12]:


num_classes = len(labels)

model_baseline = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model_baseline.summary()

d:\SE\SP\SeniorProject\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,476 (42.61 MB)

 Trainable params: 11,169,476 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
model_baseline.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [14]:
import time
import math

steps_per_epoch = math.ceil(train_generator.samples / BATCH_SIZE)
validation_steps = math.ceil(test_generator.samples / BATCH_SIZE)

start = time.time()
history_baseline = model_baseline.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    validation_data=test_generator,
    validation_steps=validation_steps,
    epochs=EPOCHS
)
train_time_sec = time.time() - start

print("Training time (sec):", round(train_time_sec, 2))

Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 2811s 31s/step - accuracy: 0.4024 - loss: 1.2588 - val_accuracy: 0.3071 - val_loss: 1.6235
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - accuracy: 0.5293 - loss: 1.0692 - val_accuracy: 0.3173 - val_loss: 2.0318
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 95s 1s/step - accuracy: 0.5976 - loss: 0.9491 - val_accuracy: 0.3376 - val_loss: 2.0964
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.6282 - loss: 0.8760 - val_accuracy: 0.3807 - val_loss: 2.5260
Epoch 5/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 116s 1s/step - accuracy: 0.6557 - loss: 0.8205 - val_accuracy: 0.3528 - val_loss: 2.4656
Epoch 6/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 284s 3s/step - accuracy: 0.6470 - loss: 0.8148 - val_accuracy: 0.3832 - val_loss: 2.6921
Epoch 7/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 98s 1s/step - accuracy: 0.6770 - loss: 0.7689 - val_accuracy: 0.3934 - val_loss: 2.4100
Epoch 8/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 97s 1s/step - accuracy: 0.6889 - loss: 0.7468 - val_accuracy: 0.4061 - va

In [15]:
baseline_loss, baseline_acc = model_baseline.evaluate(
    test_generator,
    steps=validation_steps
)
print("Baseline Loss:", baseline_loss)
print("Baseline Accuracy:", baseline_acc)

13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 416ms/step - accuracy: 0.5102 - loss: 4.0839
Baseline Loss: 4.083947658538818
Baseline Accuracy: 0.510152280330658


In [16]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# reset generator to start
test_generator.reset()

y_prob = model_baseline.predict(test_generator, steps=validation_steps)
y_pred = np.argmax(y_prob, axis=1)

y_true = test_generator.classes  # integer labels in directory order
class_names = list(test_generator.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 430ms/step
Confusion Matrix:
 [[15 25 55  5]
 [ 2 55 55  3]
 [ 2  5 98  0]
 [ 0  2 39 33]]

Classification Report:

                  precision    recall  f1-score   support

    glioma_tumor       0.79      0.15      0.25       100
meningioma_tumor       0.63      0.48      0.54       115
        no_tumor       0.40      0.93      0.56       105
 pituitary_tumor       0.80      0.45      0.57        74

        accuracy                           0.51       394
       macro avg       0.66      0.50      0.48       394
    weighted avg       0.64      0.51      0.48       394



In [17]:
os.makedirs(MODELS_DIR, exist_ok=True)

baseline_model_path = MODELS_DIR / "baseline_cnn.keras"
model_baseline.save(str(baseline_model_path))
print(f"Saved to {baseline_model_path}")

Saved to G:\My Drive\SeniorProject\datasets\dataset\models\baseline_cnn.keras


# Transfer Learning Generators

In [18]:
transfer_train_datagen= ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
)
transfer_test_datagen=ImageDataGenerator(
    preprocessing_function=preprocess_input
)
train_gen_tf=transfer_train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)
test_gen_tf=transfer_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 2870 images belonging to 4 classes.
Found 394 images belonging to 4 classes.


In [19]:
base_model=MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

X= base_model.output
X=GlobalAveragePooling2D()(X)
X=Dense(128, activation='relu')(X)
X=Dropout(0.5)(X)
outputs= Dense(train_gen_tf.num_classes, activation='softmax')(X)
model_transfer=Model(inputs=base_model.input, outputs=outputs)

# Compile

In [20]:
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [21]:
EPOCHS_TL = 20
history_transfer= model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=EPOCHS_TL
)

Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 107s 1s/step - accuracy: 0.6756 - loss: 0.8117 - val_accuracy: 0.5533 - val_loss: 1.1456
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 94s 1s/step - accuracy: 0.7920 - loss: 0.5235 - val_accuracy: 0.5838 - val_loss: 1.2181
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 107s 1s/step - accuracy: 0.8265 - loss: 0.4520 - val_accuracy: 0.5635 - val_loss: 1.1378
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 106s 1s/step - accuracy: 0.8422 - loss: 0.4226 - val_accuracy: 0.6193 - val_loss: 1.2476
Epoch 5/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - accuracy: 0.8460 - loss: 0.4085 - val_accuracy: 0.6371 - val_loss: 1.1798
Epoch 6/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - accuracy: 0.8463 - loss: 0.3800 - val_accuracy: 0.6472 - val_loss: 1.1949
Epoch 7/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.8645 - loss: 0.3515 - val_accuracy: 0.6751 - val_loss: 1.1744
Epoch 8/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.8603 - loss: 0.3600 - val_accuracy: 0.6701 - va

In [22]:
transfer_loss, transfer_acc = model_transfer.evaluate(test_gen_tf)
print("Transfer Learning Loss:", transfer_loss)

13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 636ms/step - accuracy: 0.7183 - loss: 1.4886
Transfer Learning Loss: 1.4885673522949219


In [23]:
print('Comparision:')
print('Baseline Accuracy:', baseline_acc)
print('Transfer Learning Accuracy:', transfer_acc)

Comparision:
Baseline Accuracy: 0.510152280330658
Transfer Learning Accuracy: 0.7182741165161133


In [24]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
historical_finetune= model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=10
)

Epoch 1/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 135s 1s/step - accuracy: 0.6244 - loss: 1.4510 - val_accuracy: 0.6777 - val_loss: 2.1863
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 114s 1s/step - accuracy: 0.7990 - loss: 0.6181 - val_accuracy: 0.6574 - val_loss: 2.6286
Epoch 3/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 127s 1s/step - accuracy: 0.8418 - loss: 0.4178 - val_accuracy: 0.6523 - val_loss: 2.7111
Epoch 4/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 130s 1s/step - accuracy: 0.8606 - loss: 0.3720 - val_accuracy: 0.6497 - val_loss: 2.5914
Epoch 5/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 111s 1s/step - accuracy: 0.8763 - loss: 0.3292 - val_accuracy: 0.6675 - val_loss: 2.4926
Epoch 6/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 111s 1s/step - accuracy: 0.8882 - loss: 0.3028 - val_accuracy: 0.6701 - val_loss: 2.4052
Epoch 7/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - accuracy: 0.8899 - loss: 0.2934 - val_accuracy: 0.6827 - val_loss: 2.3081
Epoch 8/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 110s 1s/step - accuracy: 0.8958 - loss: 0.2685 - val_accuracy: 0.7030 - v

In [25]:
os.makedirs(MODELS_DIR, exist_ok=True)

transfer_model_path = MODELS_DIR / "mobilenetv2_transfer.keras"
model_transfer.save(str(transfer_model_path))
print(f"Saved to {transfer_model_path}")

Saved to G:\My Drive\SeniorProject\datasets\dataset\models\mobilenetv2_transfer.keras
